<div align="center">
<h1 style="background: linear-gradient(135deg, #c0392b 0%, #e74c3c 100%); color: white; padding: 30px; border-radius: 20px; margin-bottom: 20px;">🔐 KRİPTOQRAFİYA KURSU</h1>
<h2 style="color: #2c3e50; font-size: 2.5em; margin: 20px 0;">Məşğələ 27</h2>
<h2 style="color: #e74c3c; font-size: 2em; margin: 20px 0;">Kriptoanaliz və Yan Kanal Hücumları</h2>
<h3 style="color: #7f8c8d; font-size: 1.5em; margin: 20px 0;">Şifrələri Sındırma Sənəti</h3>
<br>
<div style="background-color: #e74c3c; color: white; padding: 15px; border-radius: 50px; display: inline-block; font-size: 1.2em;">⏱️ Məşğələ vaxtı: 2 saat</div>
<br><br>
<p><em>Hazırlanma tarixi: 2024 | Yenilənmiş versiya</em></p>
</div>

## 📑 Mündəricat

1. İcra müqaviləsi və kriptoanaliz sərhədi
2. Exhaustive search, differential və linear toy analiz
3. Timing leakage və `compare_digest`
4. Synthetic power traces və masking correctness
5. EM, acoustic, cold-boot və cache sərhədləri
6. CRT-RSA fault recovery
7. Self-check, tapşırıq və mənbələr


## 🎯 Məqsədlər

Tələbə toy analiz ilə real kriptoanalizi ayırır; DDT/LAT hesablayır; secret-dependent operation trace-i müşahidə edir; Python timing nəticəsini constant-time sübutu saymır; masking üçün `unmask(masked_f(x)) == f(x)` xassəsini exhaustive yoxlayır; cache/fault modellərinin ön şərtlərini göstərir.


## 📚 Asılılıqlar

Yalnız Python standard library istifadə olunur. Şəbəkə, paket quraşdırılması, fayl, `input()` və real sistemə hücum yoxdur.


In [ ]:
import sys
print(f"Standard-library-only mühit: Python {sys.version_info.major}.{sys.version_info.minor}")


## 🔧 Hazırlıq

Bütün hücumlar kiçik, icazəli və sintetik modellərdir. Statistik korrelyasiya səbəbiyyət və real cihaz exploitability sübutu deyil. Production mitigation audit edilmiş constant-time/masked implementation və platforma ölçmələri tələb edir.


### İcra prinsipi

Hər nəticə assertion ilə təsdiqlənir; uğursuz nəticə “uğur” kimi çap edilmir. Random noise əvəzinə reproduktiv deterministik nümunələr seçilir.


In [ ]:
import hmac
import math
import random
import statistics
from collections import Counter

RNG = random.Random(29)
assert hmac.compare_digest and math.gcd
print('Məcburi standard-library API-lər hazırdır')


### Təhlükəsizlik sərhədi

Notebook operation count və synthetic leakage göstərir. CPython interpreter, bigint və memory access constant-time deyil; burada yazılmış funksiya production secret emalı üçün istifadə olunmamalıdır.


In [ ]:
LAB_CONTRACT={
 'external_target':False,'real_exploit':False,'constant_time_claim':False,
 'masking_correctness_required':True,'negative_tests_required':True}
assert not any(LAB_CONTRACT[k] for k in ('external_target','real_exploit','constant_time_claim'))
assert LAB_CONTRACT['masking_correctness_required'] and LAB_CONTRACT['negative_tests_required']
print('Kriptoanaliz laboratoriya müqaviləsi təsdiqləndi')


## 📖 Kriptoanaliz nədir?

Kriptoanaliz konkret modeldə primitive/protocol təhlükəsizlik xassəsini qiymətləndirir. Side-channel və fault attacks riyazi primitive-dən əlavə implementation, cihaz, ölçmə və attacker-access ön şərtlərinə malikdir.


### Evidence pillələri

Toy counterexample, synthetic simulation, controlled lab measurement və real target validation eyni sübut səviyyəsi deyil. Nəticə yalnız faktiki sınaq əhatəsinə aid edilir.


## 🔄 Tam axtarış

Exhaustive search mümkün açar sahəsini yoxlayır. Bir toy 8-bit açarın tapılması production key length üçün praktik nəticə vermir; mürəkkəblik açar sahəsi və hər yoxlamanın qiymətindən asılıdır.


### 4.1 8-bit XOR toy modeli

Known plaintext/ciphertext cütü yanlış açarları ayırmaq üçün kifayət etməyə bilər; bir neçə cüt istifadə olunur.


In [ ]:
def recover_xor_key(pairs):
    candidates=[k for k in range(256) if all((p^k)==c for p,c in pairs)]
    return candidates

toy_key=0xA7
known_pairs=[(p,p^toy_key) for p in (0x00,0x35,0xFF)]
bruteforce_candidates=recover_xor_key(known_pairs)
assert bruteforce_candidates==[toy_key]
assert recover_xor_key([(0,0),(1,0)])==[]
print({'tested_keys':256,'recovered':True,'toy_only':True})


### ✍️ Çalışma 1 — 1 bal

4, 8 və 12-bit toy key spaces üçün yoxlama sayını ölçün. **Meyar:** eyni oracle, bütün açarlar, doğru/yanlış/no-solution tests və production nəticəsi çıxarmamaq.


## 📊 Diferensial kriptoanaliz

DDT seçilmiş input difference üçün output differences paylanmasını sayır. Bir S-box DDT-si bütöv block cipher attack complexity-ni təkbaşına müəyyən etmir; round structure, key schedule və trail birləşməsi ayrıca lazımdır.


### 5.1 4-bit S-box DDT

Hər DDT sətrinin cəmi 16 olmalıdır; `Δx=0` yalnız `Δy=0` verir.


In [ ]:
SBOX=(0xC,5,6,0xB,9,0,0xA,0xD,3,0xE,0xF,8,4,7,1,2)
def differential_distribution_table(sbox):
    n=len(sbox); return [[sum(1 for x in range(n) if (sbox[x]^sbox[x^dx])==dy) for dy in range(n)] for dx in range(n)]

ddt=differential_distribution_table(SBOX)
assert all(sum(row)==16 for row in ddt)
assert ddt[0][0]==16 and sum(ddt[0][1:])==0
max_nontrivial=max(max(row) for row in ddt[1:])
assert max_nontrivial==4
print({'rows':16,'row_sum':16,'max_nontrivial':max_nontrivial})


### ✍️ Çalışma 2 — 1 bal

İki 4-bit permutation S-box üçün maximum nontrivial DDT entry müqayisə edin. **Meyar:** exhaustive table, row invariants və yalnız S-box səviyyəli nəticə.


## 📈 Xətti kriptoanaliz

LAT input/output mask parity münasibətinin balansdan sapmasını hesablayır. Bias sıfırdan fərqli olduqda correlation mövcuddur; bu, avtomatik key recovery və ya bütöv cipher zəifliyi deyil.


In [ ]:
def parity(value): return value.bit_count()&1
def linear_approximation_table(sbox):
    n=len(sbox)
    return [[sum(1 if parity(a&x)==parity(b&sbox[x]) else -1 for x in range(n)) for b in range(n)] for a in range(n)]

lat=linear_approximation_table(SBOX)
assert len(lat)==16 and all(len(row)==16 for row in lat)
assert lat[0][0]==16 and all(v==0 for v in lat[0][1:])
max_linear=max(abs(lat[a][b]) for a in range(1,16) for b in range(1,16))
assert max_linear>0
print({'max_nontrivial_correlation_sum':max_linear,'samples_per_entry':16})


### ✍️ Çalışma 3 — 1 bal

Ən böyük absolute nontrivial LAT entry-ni və maskalarını tapın. **Meyar:** trivial masks çıxarılır, sign saxlanılır və correlation-to-attack sərhədi izah olunur.


## ⏱️ Timing leakage

Secret-dependent branch və memory access timing leakage mənbəyi ola bilər. Python timing experiment constant-time property sübut etmir; scheduler, interpreter, allocation, bigint və CPU effects qarışır.


### 6.1 Square-and-multiply operation trace

Vaxtı saxtalaşdırmaq üçün `sleep` istifadə olunmur. Bunun əvəzinə secret exponent bit-lərinin yaratdığı operation trace açıq göstərilir.


In [ ]:
def square_multiply_trace(base, exponent, modulus):
    result=1; trace=[]
    for bit in bin(exponent)[2:]:
        result=(result*result)%modulus; trace.append('S')
        if bit=='1':
            result=(result*base)%modulus; trace.append('M')
    return result,''.join(trace)

trace_result,operation_trace=square_multiply_trace(7,0b101101,101)
assert trace_result==pow(7,0b101101,101)
assert operation_trace.count('M')==(0b101101).bit_count()
print({'trace':operation_trace,'production_constant_time':False})


In [ ]:
def pearson(xs,ys):
    mx,my=statistics.mean(xs),statistics.mean(ys)
    num=sum((x-mx)*(y-my) for x,y in zip(xs,ys))
    den=math.sqrt(sum((x-mx)**2 for x in xs)*sum((y-my)**2 for y in ys))
    return num/den if den else 0.0

secret_bits=[(i*7+3)&1 for i in range(200)]
synthetic_times=[100+9*bit+((i%5)-2) for i,bit in enumerate(secret_bits)]
timing_correlation=pearson(secret_bits,synthetic_times)
assert timing_correlation>0.9
print({'synthetic_samples':200,'correlation':round(timing_correlation,3),'real_target_claim':False})


In [ ]:
def functional_digest_compare(a,b): return hmac.compare_digest(a,b)
assert functional_digest_compare(b'abc',b'abc')
assert not functional_digest_compare(b'abc',b'abd')
assert not functional_digest_compare(b'abc',b'abcd')
print({'api':'hmac.compare_digest','functional_tests':3,'timing_proof':False})


### 6.2 Prefix compare leakage

Early-return compare işlənmiş prefix uzunluğunu data-dependent edir. Deterministik work counter bu sızıntı mənbəyini göstərir; remote timing exploitability ayrıca noise model və çoxsaylı ölçmə tələb edir.


In [ ]:
def insecure_prefix_compare(a,b):
    work=0
    if len(a)!=len(b): return False,work
    for left,right in zip(a,b):
        work+=1
        if left!=right: return False,work
    return True,work

secret=b'SECRET42'
prefix_work=[insecure_prefix_compare(secret, secret[:i]+b'X'+secret[i+1:])[1] for i in range(len(secret))]
assert prefix_work==list(range(1,len(secret)+1))
assert insecure_prefix_compare(secret,secret)==(True,len(secret))
print({'mismatch_work':prefix_work,'remote_attack_demonstrated':False})


### ✍️ Çalışma 4 — 1 bal

Randomized order ilə ən az 1000 `perf_counter_ns` ölçməsi planlayın. **Meyar:** median/distribution, warm-up, no single-sample conclusion və `compare_digest` timing proof kimi təqdim edilmir.


## ⚡ Power analysis

SPA bir trace-də operation pattern, statistical power analysis isə çox trace ilə leakage model correlation/difference istifadə edə bilər. Synthetic trace real probe bandwidth, alignment və noise şəraitini sübut etmir.


In [ ]:
def synthetic_spa_trace(exponent):
    return ['square' if symbol=='S' else 'multiply' for symbol in square_multiply_trace(3,exponent,257)[1]]
spa_trace=synthetic_spa_trace(0b110101)
assert spa_trace.count('multiply')==(0b110101).bit_count()
print({'operations':len(spa_trace),'synthetic':True})


In [ ]:
def hamming_weight(v): return v.bit_count()
true_subkey=0x9
plaintexts=list(range(16))*20
leakage=[hamming_weight(SBOX[p^true_subkey])+((i%3)-1)*0.05 for i,p in enumerate(plaintexts)]
scores={guess:abs(pearson([hamming_weight(SBOX[p^guess]) for p in plaintexts],leakage)) for guess in range(16)}
recovered_subkey=max(scores,key=scores.get)
assert recovered_subkey==true_subkey
print({'traces':len(plaintexts),'recovered_toy_subkey':True,'method':'synthetic correlation'})


In [ ]:
def masked_xor_linear(x,key,mask):
    x0=mask; x1=x^mask
    y0=x0^key; y1=x1
    return y0,y1

masking_cases=0
for x in range(256):
    for mask in (0,1,0x55,0xAA,0xFF):
        y0,y1=masked_xor_linear(x,0x3C,mask)
        assert (y0^y1)==(x^0x3C)
        masking_cases+=1
print({'boolean_linear_cases':masking_cases,'nonlinear_claim':False})


### 7.3 Masked S-box table

Nonlinear S-box üçün sadəcə input-u XOR-mask edib adi S-box çağırmaq düzgün deyil. Toy table `T[x⊕rin] = S[x]⊕rout` kimi maskalara uyğun yenidən qurulur. Bu functional correctness-dir, side-channel security proof deyil.


In [ ]:
def build_masked_sbox(sbox,input_mask,output_mask):
    table=[0]*len(sbox)
    for x in range(len(sbox)): table[x^input_mask]=sbox[x]^output_mask
    return table

masked_sbox_cases=0
for input_mask in range(16):
    for output_mask in range(16):
        table=build_masked_sbox(SBOX,input_mask,output_mask)
        for x in range(16):
            assert (table[x^input_mask]^output_mask)==SBOX[x]
            masked_sbox_cases+=1
assert masked_sbox_cases==4096
print({'masked_sbox_cases':masked_sbox_cases,'leakage_proof':False})


### ✍️ Çalışma 5 — 1 bal

Masking üçün bütün input/mask-ları exhaustive yoxlayın. **Meyar:** correctness invariant, fresh-mask/lifecycle müzakirəsi və functional test-in side-channel proof olmadığı qeyd olunur.


## 📡 EM analysis

EM leakage yerləşmə və tezlik üzrə daha lokal ola bilər; nəticə probe, məsafə, shielding, trigger, alignment, filtering və target implementation-dan asılıdır. Synthetic power trace EM nəticəsi deyil.


## 🔊 Acoustic analysis

Akustik leakage mexaniki/elektrik komponentlərin secret-dependent fəaliyyəti ilə əlaqəli ola bilər. Konkret attack üçün cihaz, məsafə, signal processing, chosen-input access və təkrar sayı göstərilməlidir; universal “mikrofon açarı oxuyur” iddiası yoxdur.


## ❄️ Cold boot

Cold-boot attacks remanence və fiziki access ön şərtlərinə malikdir. Memory encryption, key zeroization, suspend policy, boot chain və physical controls riskə təsir edir; RAM key scan sadə string search deyil.


## 🗃️ Cache attacks

Prime+Probe shared cache set occupancy-ni, Flush+Reload isə shared memory line timing-ni istifadə edə bilər. Attacker co-location, sharing, cache geometry və scheduler ön şərtləri ayrıca qiymətləndirilir.


In [ ]:
def cache_set(address,line_size=64,sets=64): return (address//line_size)%sets
primed={cache_set(a) for a in range(0,64*64,64)}
victim_addresses=[0x1080,0x20C0,0x3100]
victim_sets={cache_set(a) for a in victim_addresses}
probe_misses=sorted(primed & victim_sets)
assert probe_misses==sorted(victim_sets)
print({'synthetic_sets':probe_misses,'real_cache_timing':False})


In [ ]:
def lookup_leakage_model(table,index): return table[index],index
value,accessed_index=lookup_leakage_model(SBOX,7)
assert value==SBOX[7] and accessed_index==7
print({'secret_dependent_index_visible_in_model':True,'aes_attack_implemented':False})


### ✍️ Çalışma 6 — 1 bal

Prime+Probe və Flush+Reload üçün ayrı prerequisite cədvəli qurun. **Meyar:** shared-memory fərqi, cache geometry və toy set modelinin real timing attack olmadığı göstərilir.


## 💥 Fault attacks

Fault timing/location/model konkret attack üçün vacibdir. CRT-RSA-da bir branch signature fault-u və düzgün signature fərqi modulus faktorunu GCD ilə aça bilər; verification-before-release və recomputation müdafiə qatlarıdır.


In [ ]:
def crt_rsa_signature(message,p,q,d,fault=False):
    sp=pow(message,d%(p-1),p); sq=pow(message,d%(q-1),q)
    if fault: sp=(sp+1)%p
    h=(pow(q,-1,p)*(sp-sq))%p
    return (sq+q*h)%(p*q)

p,q,e=61,53,17
phi=(p-1)*(q-1); d=pow(e,-1,phi); n=p*q; message=42
good=crt_rsa_signature(message,p,q,d)
faulty=crt_rsa_signature(message,p,q,d,True)
recovered_factor=math.gcd((good-faulty)%n,n)
assert pow(good,e,n)==message and pow(faulty,e,n)!=message
assert recovered_factor in (p,q) and recovered_factor not in (1,n)
print({'fault_detected':True,'factor_recovered':recovered_factor,'toy_rsa':True})


## 🛡️ Müdafiə strategiyası

Audit edilmiş constant-time primitives, masking/hiding, blinding, redundancy/fault detection, key isolation, cache partitioning və physical controls threat model üzrə seçilir. Compiler, interpreter, microarchitecture və integration nəticəni dəyişə bilər; source-level “branch yoxdur” kifayət deyil.


### ✍️ Çalışma 7 — 1 bal

Timing, power/cache və fault üçün asset→capability→leakage/fault model→evidence→mitigation cədvəli qurun. **Meyar:** hər mitigation-in scope və residual risk-i var.


## 🖥️ İnteqrasiya edilmiş self-check

Self-check yalnız toy funksional/statistik invariantları təsdiqləyir; real hardware assurance vermir.


In [ ]:
def run_side_channel_self_check():
    checks={'bruteforce':bruteforce_candidates==[toy_key],'ddt':max_nontrivial==4,'lat':max_linear>0,
    'operation_trace':trace_result==pow(7,0b101101,101),'synthetic_timing':timing_correlation>0.9,
    'compare_digest':functional_digest_compare(b'a',b'a'),'toy_correlation':recovered_subkey==true_subkey,
    'linear_masking':masking_cases==1280,'masked_sbox':masked_sbox_cases==4096,
    'cache_model':bool(probe_misses),'crt_fault':recovered_factor in (p,q)}
    assert all(checks.values()); return checks
side_channel_checks=run_side_channel_self_check()
print({'passed':sum(side_channel_checks.values()),'total':len(side_channel_checks)})


## 🏠 Ev tapşırığı — 10 bal

İcazəli synthetic leakage study hazırlayın: threat/evidence model 2 bal; reproduktiv traces 2; yanlış və doğru hypothesis 2; statistical method 1; exhaustive functional checks 1; mitigation scope 1; limitations 1. Real target və private data qadağandır.


## 📌 Yekun

- Toy DDT/LAT bütöv cipher attack deyil.
- Python timing constant-time proof deyil.
- Masking əvvəlcə funksiyanı bütün input-larda saxlamalıdır.
- Functional correctness leakage resistance sübutu deyil.
- Side-channel/fault nəticəsi capability, target və ölçmə modelinə bağlıdır.
- CRT fault toy nümunəsi real faktor recovery göstərir.


## 📚 Mənbələr

- NIST SP 800-56B Rev. 2 — RSA implementation assurance konteksti
- NIST FIPS 140-3 — non-invasive/fault mitigation assurance konteksti
- Kocher 1996 — Timing Attacks
- Mangard, Oswald, Popp — Power Analysis Attacks
- Boneh, DeMillo, Lipton — CRT computation faults
- Python `hmac.compare_digest` documentation

Primary research nəticələri yalnız uyğun attack model və implementation scope ilə tətbiq olunur.
